In [ ]:
# Verify saved files
import os

print("Verifying saved files...")
saved_files = ['best_diabetes_model.pkl', 'scaler.pkl', 'le_gender.pkl', 'le_smoking.pkl']

for file in saved_files:
    if os.path.exists(file):
        size = os.path.getsize(file)
        print(f"✓ {file} ({size} bytes)")
    else:
        print(f"✗ {file} - NOT FOUND")

print("\nModel Training Pipeline Complete!")
print("The trained model is ready for deployment in the Streamlit app.")

In [ ]:
# Save the best model and preprocessing objects
print("Saving models and preprocessing objects...")

# Save the best XGBoost model
joblib.dump(best_xgb_model, 'best_diabetes_model.pkl')
print("✓ Best model saved as 'best_diabetes_model.pkl'")

# Save the scaler
joblib.dump(scaler, 'scaler.pkl')
print("✓ StandardScaler saved as 'scaler.pkl'")

# Save the label encoders
joblib.dump(le_gender, 'le_gender.pkl')
print("✓ Gender LabelEncoder saved as 'le_gender.pkl'")

joblib.dump(le_smoking, 'le_smoking.pkl')
print("✓ Smoking History LabelEncoder saved as 'le_smoking.pkl'")

print("\n" + "="*50)
print("All models and preprocessing objects saved successfully!")
print("="*50)

## 10. Save the Trained Model

Serialize and save the final trained model using joblib for future predictions.

In [ ]:
# Visualize confusion matrix and ROC curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=False)
axes[0].set_title('Confusion Matrix - Best XGBoost Model', fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_test_pred_prob)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {best_roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve - Best XGBoost Model', fontweight='bold')
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate best model on test set
best_xgb_model = grid_search.best_estimator_

y_test_pred_best = best_xgb_model.predict(X_test_scaled)
y_test_pred_prob = best_xgb_model.predict_proba(X_test_scaled)[:, 1]

# Calculate metrics
best_accuracy = accuracy_score(y_test, y_test_pred_best)
best_precision = precision_score(y_test, y_test_pred_best)
best_recall = recall_score(y_test, y_test_pred_best)
best_f1 = f1_score(y_test, y_test_pred_best)
best_roc_auc = roc_auc_score(y_test, y_test_pred_prob)

print("Best XGBoost Model Performance on Test Set:")
print(f"Accuracy: {best_accuracy:.4f}")
print(f"Precision: {best_precision:.4f}")
print(f"Recall: {best_recall:.4f}")
print(f"F1-Score: {best_f1:.4f}")
print(f"ROC-AUC: {best_roc_auc:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred_best)
print("\nConfusion Matrix:")
print(cm)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred_best))

In [ ]:
# Best model is XGBoost based on evaluation results
# Hyperparameter tuning for XGBoost
print("Starting Hyperparameter Tuning for XGBoost...")

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0]
}

# Use smaller param grid for faster execution
param_grid_reduced = {
    'n_estimators': [100, 200],
    'max_depth': [5, 7],
    'learning_rate': [0.1, 0.2]
}

# Grid Search
grid_search = GridSearchCV(
    XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'),
    param_grid_reduced,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

print("Fitting GridSearchCV...")
grid_search.fit(X_train_scaled, y_train)

print("\nBest Parameters:", grid_search.best_params_)
print("Best Cross-Validation F1-Score:", grid_search.best_score_)

## 9. Hyperparameter Tuning

Use GridSearchCV to find optimal hyperparameters for the best performing model (XGBoost).

In [ ]:
# Visualize model performance
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Test Accuracy comparison
axes[0, 0].bar(results_df['Model'], results_df['Test Accuracy'], color='skyblue', edgecolor='black')
axes[0, 0].set_title('Test Accuracy Comparison', fontweight='bold')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].set_ylim([0, 1])
for i, v in enumerate(results_df['Test Accuracy']):
    axes[0, 0].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

# Precision comparison
axes[0, 1].bar(results_df['Model'], results_df['Precision'], color='lightcoral', edgecolor='black')
axes[0, 1].set_title('Precision Comparison', fontweight='bold')
axes[0, 1].set_ylabel('Precision')
axes[0, 1].set_ylim([0, 1])
for i, v in enumerate(results_df['Precision']):
    axes[0, 1].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

# Recall comparison
axes[1, 0].bar(results_df['Model'], results_df['Recall'], color='lightgreen', edgecolor='black')
axes[1, 0].set_title('Recall Comparison', fontweight='bold')
axes[1, 0].set_ylabel('Recall')
axes[1, 0].set_ylim([0, 1])
for i, v in enumerate(results_df['Recall']):
    axes[1, 0].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

# F1-Score comparison
axes[1, 1].bar(results_df['Model'], results_df['F1-Score'], color='gold', edgecolor='black')
axes[1, 1].set_title('F1-Score Comparison', fontweight='bold')
axes[1, 1].set_ylabel('F1-Score')
axes[1, 1].set_ylim([0, 1])
for i, v in enumerate(results_df['F1-Score']):
    axes[1, 1].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Function to evaluate model performance
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Metrics
    train_accuracy = accuracy_score(y_train, y_train_pred)
    test_accuracy = accuracy_score(y_test, y_test_pred)
    precision = precision_score(y_test, y_test_pred)
    recall = recall_score(y_test, y_test_pred)
    f1 = f1_score(y_test, y_test_pred)
    
    # Cross-validation score
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    
    return {
        'Model': model_name,
        'Train Accuracy': train_accuracy,
        'Test Accuracy': test_accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'CV Mean': cv_scores.mean(),
        'CV Std': cv_scores.std()
    }

# Evaluate all models
results = []
for name, model in models.items():
    result = evaluate_model(model, X_train_scaled, X_test_scaled, y_train, y_test, name)
    results.append(result)

# Create results dataframe
results_df = pd.DataFrame(results)
print("Model Evaluation Results:")
print(results_df.to_string(index=False))

## 8. Evaluate Model Performance

Assess models using metrics like accuracy, precision, recall, F1-score, and cross-validation scores.

In [ ]:
# Dictionary to store models
models = {}

# 1. Logistic Regression
print("Training Logistic Regression...")
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)
models['Logistic Regression'] = lr_model
print("✓ Logistic Regression trained")

# 2. Decision Tree Classifier
print("Training Decision Tree Classifier...")
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train_scaled, y_train)
models['Decision Tree'] = dt_model
print("✓ Decision Tree trained")

# 3. Random Forest Classifier
print("Training Random Forest Classifier...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)
models['Random Forest'] = rf_model
print("✓ Random Forest trained")

# 4. XGBoost Classifier
print("Training XGBoost Classifier...")
xgb_model = XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False, eval_metric='logloss')
xgb_model.fit(X_train_scaled, y_train)
models['XGBoost'] = xgb_model
print("✓ XGBoost trained")

print("\nAll models trained successfully!")

## 7. Train Machine Learning Models

Train multiple machine learning models such as Logistic Regression, Decision Trees, Random Forests, and XGBoost.

In [ ]:
# Scale the features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled training features shape:", X_train_scaled.shape)
print("Scaled testing features shape:", X_test_scaled.shape)

In [ ]:
# Split data into training and testing sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)
print("\nTraining set target distribution:")
print(y_train.value_counts())
print("\nTesting set target distribution:")
print(y_test.value_counts())

## 6. Split Data into Train and Test Sets

Use train_test_split to divide data into training and testing subsets with an appropriate ratio.

In [ ]:
# Separate features and target
X = df.drop('diabetes', axis=1)
y = df['diabetes']

# Initialize label encoders for categorical variables
le_gender = LabelEncoder()
le_smoking = LabelEncoder()

# Encode categorical variables
if 'gender' in X.columns:
    X['gender'] = le_gender.fit_transform(X['gender'])
if 'smoking_history' in X.columns:
    X['smoking_history'] = le_smoking.fit_transform(X['smoking_history'])

print("Features after encoding:")
print(X.head())
print("\nFeature columns:", X.columns.tolist())
print("Target distribution:")
print(y.value_counts())

## 5. Feature Engineering

Create new features, scale/normalize existing features, and encode categorical variables.

In [ ]:
# Correlation matrix
plt.figure(figsize=(12, 8))
numeric_df = df.select_dtypes(include=['int64', 'float64'])
correlation_matrix = numeric_df.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f', square=True)
plt.title('Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of numerical features
plt.figure(figsize=(15, 10))
for i, col in enumerate(numerical_cols, 1):
    plt.subplot(3, 3, i)
    df[col].hist(bins=30, edgecolor='black', color='skyblue')
    plt.title(f'Distribution of {col}', fontweight='bold')
    plt.xlabel(col)
    plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Visualize target variable distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
df['diabetes'].value_counts().plot(kind='bar', ax=axes[0], color=['skyblue', 'salmon'])
axes[0].set_title('Diabetes Distribution (Count)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Diabetes')
axes[0].set_ylabel('Count')

# Pie chart
df['diabetes'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['skyblue', 'salmon'])
axes[1].set_title('Diabetes Distribution (Percentage)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

## 4. Exploratory Data Analysis (EDA)

Visualize distributions, correlations, and relationships between features using plots and statistical summaries.

In [ ]:
# Check for duplicates
print("Duplicate Rows:", df.duplicated().sum())

# Remove duplicates if any
df = df.drop_duplicates()
print("Dataset Shape After Removing Duplicates:", df.shape)

# Check data types
print("\nData Types:")
print(df.dtypes)

# Identify categorical and numerical columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("\nCategorical Columns:", categorical_cols)
print("Numerical Columns:", numerical_cols)

## 3. Data Preprocessing and Cleaning

Handle missing values, remove duplicates, and perform data type conversions as needed.

In [ ]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())
print("\nMissing Values Percentage:")
print((df.isnull().sum() / len(df)) * 100)

# Check target variable distribution
print("\n" + "="*50)
print("Target Variable Distribution:")
print(df['diabetes'].value_counts())
print("\nTarget Variable Percentage:")
print(df['diabetes'].value_counts(normalize=True) * 100)

In [ ]:
# Load the dataset
df = pd.read_csv('diabetes_prediction_dataset.csv')

# Display basic information
print("Dataset Shape:", df.shape)
print("\n" + "="*50)
print("First Few Rows:")
print(df.head())
print("\n" + "="*50)
print("Dataset Info:")
print(df.info())
print("\n" + "="*50)
print("Statistical Summary:")
print(df.describe())

## 2. Load and Explore Data

Load the dataset and perform initial exploration to understand its structure, size, and basic statistics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, roc_curve
import joblib
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Import Required Libraries

Import necessary libraries including pandas, NumPy, scikit-learn, matplotlib, and seaborn for data manipulation, analysis, and visualization.

# Diabetes Prediction — Complete ML Pipeline

**Overview:** This notebook demonstrates a complete machine learning workflow for diabetes prediction, from exploratory data analysis to model training, evaluation, and deployment.